# Treino XGBoost com Optuna - Detecção de Fadiga

Treina modelo XGBoost para classificação binária: Alerta vs Sonolento
Utiliza Optuna para otimização de hiperparâmetros

In [1]:
import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from scipy import stats
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas")

Bibliotecas carregadas


/home/zhizhu/tcc/ml-fadiga-motoristas/venv_ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Carrega dados processados
dados_dir = Path("processed_data_uta_rldd_CORRECTED")
arquivo_X = dados_dir / "X_sequences.npy"
arquivo_y = dados_dir / "y_labels.npy"

if not arquivo_X.exists() or not arquivo_y.exists():
    print("Erro: Execute primeiro o notebook de preparação dos dados")
    print(f"Arquivos necessários:")
    print(f"  {arquivo_X}")
    print(f"  {arquivo_y}")
    raise FileNotFoundError("Arquivos de dados não encontrados")

# Carrega sequences e labels
X_sequences = np.load(arquivo_X)
y_labels = np.load(arquivo_y)

print(f"Dados carregados: {X_sequences.shape[0]} amostras")
print(f"Shape das sequences: {X_sequences.shape}")
print(f"Labels únicas: {np.unique(y_labels)}")

Dados carregados: 32111 amostras
Shape das sequences: (32111, 90, 4)
Labels únicas: [ 0  5 10]


In [3]:
# Converte para classificação binária
# 0,5 -> 0 (Alerta), 10 -> 1 (Sonolento)
y_binario = np.where(y_labels == 10, 1, 0)

print("Distribuição original:")
unique, counts = np.unique(y_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count} amostras")

print("\nDistribuição binária:")
unique, counts = np.unique(y_binario, return_counts=True)
labels_texto = ['Alerta', 'Sonolento']
for label, count, texto in zip(unique, counts, labels_texto):
    print(f"  {texto} ({label}): {count} amostras")

Distribuição original:
  0: 10826 amostras
  5: 10473 amostras
  10: 10812 amostras

Distribuição binária:
  Alerta (0): 21299 amostras
  Sonolento (1): 10812 amostras


In [4]:
# Extrai features estatísticas das sequences
class ExtratorFeatures:
    def __init__(self):
        nomes_sinais = ['PERCLOS', 'MAR', 'BLINK_RATE', 'HEAD_STABILITY']
        nomes_stats = ['mean', 'std', 'median', 'min', 'max', 'range', 'q25', 'q75', 'trend', 'zcr', 'autocorr']
        
        self.feature_names = []
        for sinal in nomes_sinais:
            for stat in nomes_stats:
                self.feature_names.append(f"{sinal}_{stat}")
    
    def trend_slope(self, sinal):
        if len(sinal) < 2:
            return 0.0
        x = np.arange(len(sinal))
        try:
            slope, _, _, _, _ = stats.linregress(x, sinal)
            return slope if not np.isnan(slope) else 0.0
        except:
            return 0.0
    
    def zero_crossing_rate(self, sinal):
        if len(sinal) < 2:
            return 0.0
        mean_centered = sinal - np.mean(sinal)
        crossings = np.sum(np.diff(np.sign(mean_centered)) != 0)
        return crossings / len(sinal)
    
    def autocorr_lag1(self, sinal):
        if len(sinal) < 3:
            return 0.0
        try:
            corr = np.corrcoef(sinal[:-1], sinal[1:])[0, 1]
            return corr if not np.isnan(corr) else 0.0
        except:
            return 0.0
    
    def extrair_features_sinal(self, sinal):
        features = [
            np.mean(sinal), np.std(sinal), np.median(sinal),
            np.min(sinal), np.max(sinal), np.ptp(sinal),
            np.percentile(sinal, 25), np.percentile(sinal, 75),
            self.trend_slope(sinal), self.zero_crossing_rate(sinal), self.autocorr_lag1(sinal)
        ]
        return features
    
    def transform(self, X_sequences):
        n_samples = X_sequences.shape[0]
        n_features = len(self.feature_names)
        X_features = np.zeros((n_samples, n_features))
        
        for i in range(n_samples):
            sequence = X_sequences[i]
            sample_features = []
            
            for signal_idx in range(4):
                sinal = sequence[:, signal_idx]
                features_sinal = self.extrair_features_sinal(sinal)
                sample_features.extend(features_sinal)
            
            X_features[i] = sample_features
        
        return X_features

extrator = ExtratorFeatures()
print(f"Extrator criado - {len(extrator.feature_names)} features por amostra")

Extrator criado - 44 features por amostra


In [5]:
# Extrai features e divide dados
print("Extraindo features...")
X_features = extrator.transform(X_sequences)
print(f"Features extraídas: {X_features.shape}")

# Divide treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_binario, test_size=0.2, random_state=42, stratify=y_binario
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

# Divide treino em treino/validação para Optuna
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Treino para otimização: {X_train_opt.shape[0]} amostras")
print(f"Validação para otimização: {X_val_opt.shape[0]} amostras")

Extraindo features...
Features extraídas: (32111, 44)
Treino: 25688 amostras
Teste: 6423 amostras
Treino para otimização: 20550 amostras
Validação para otimização: 5138 amostras


In [6]:
# Normaliza features
scaler = StandardScaler()
X_train_opt_scaled = scaler.fit_transform(X_train_opt)
X_val_opt_scaled = scaler.transform(X_val_opt)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features normalizadas")

Features normalizadas


In [7]:
# Calcula pesos das classes
unique, counts = np.unique(y_train, return_counts=True)
total = len(y_train)
peso_alerta = total / (2 * counts[0]) if 0 in unique else 1.0
peso_sonolento = total / (2 * counts[1]) if 1 in unique else 1.0
scale_pos_weight = peso_sonolento / peso_alerta

print(f"Peso Alerta: {peso_alerta:.2f}")
print(f"Peso Sonolento: {peso_sonolento:.2f}")
print(f"Scale pos weight: {scale_pos_weight:.2f}")

Peso Alerta: 0.75
Peso Sonolento: 1.49
Scale pos weight: 1.97


In [8]:
# Define função objetivo para Optuna
def objective(trial):
    # Sugere hiperparâmetros
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    # Treina modelo
    modelo = xgb.XGBClassifier(**params)
    modelo.fit(X_train_opt_scaled, y_train_opt)
    
    # Avalia no conjunto de validação
    y_pred_proba = modelo.predict_proba(X_val_opt_scaled)[:, 1]
    auc_score = roc_auc_score(y_val_opt, y_pred_proba)
    
    return auc_score

print("Função objetivo definida")

Função objetivo definida


In [9]:
# Executa otimização com Optuna
print("Iniciando otimização de hiperparâmetros com Optuna...")

# Cria estudo
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42)
)

# Executa otimização
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nOtimização concluída!")
print(f"Melhor AUC: {study.best_value:.4f}")
print(f"Melhores hiperparâmetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2025-09-14 18:26:23,970] A new study created in memory with name: no-name-dedaf63c-c051-495e-aca3-3b6054f183ba


Iniciando otimização de hiperparâmetros com Optuna...


Best trial: 0. Best value: 0.968881:   1%|█▏                                                                                                                  | 1/100 [00:00<01:13,  1.35it/s]

[I 2025-09-14 18:26:24,707] Trial 0 finished with value: 0.9688809397812694 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.22227824312530747, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 0.15599452033620265, 'reg_lambda': 0.05808361216819946}. Best is trial 0 with value: 0.9688809397812694.


Best trial: 0. Best value: 0.968881:   2%|██▎                                                                                                                 | 2/100 [00:03<02:42,  1.66s/it]

...

[I 2025-09-14 18:32:08,508] Trial 97 finished with value: 0.9729217550001356 and parameters: {'n_estimators': 475, 'max_depth': 10, 'learning_rate': 0.03705014622853836, 'subsample': 0.9770073849597474, 'colsample_bytree': 0.6311748257707774, 'reg_alpha': 0.9989613278901015, 'reg_lambda': 0.04017830320917812}. Best is trial 96 with value: 0.9758902887459633.


Best trial: 96. Best value: 0.97589:  99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊ | 99/100 [05:51<00:06,  6.41s/it]

[I 2025-09-14 18:32:15,663] Trial 98 finished with value: 0.9750761553909197 and parameters: {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.06849891037592437, 'subsample': 0.9179451522643818, 'colsample_bytree': 0.6553498229810898, 'reg_alpha': 0.9767642709870352, 'reg_lambda': 0.08409600496170366}. Best is trial 96 with value: 0.9758902887459633.


Best trial: 96. Best value: 0.97589: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [05:56<00:00,  3.57s/it]

[I 2025-09-14 18:32:20,884] Trial 99 finished with value: 0.9754847485684822 and parameters: {'n_estimators': 474, 'max_depth': 10, 'learning_rate': 0.05454260334426573, 'subsample': 0.9704185701389625, 'colsample_bytree': 0.6594731218436674, 'reg_alpha': 0.8860911159519915, 'reg_lambda': 0.007329234874282168}. Best is trial 96 with value: 0.9758902887459633.

Otimização concluída!
Melhor AUC: 0.9759
Melhores hiperparâmetros:
  n_estimators: 490
  max_depth: 10
  learning_rate: 0.05469030015305462
  subsample: 0.9762684373431535
  colsample_bytree: 0.6312657427751117
  reg_alpha: 0.9686376502670134
  reg_lambda: 0.0002969207856315659


In [10]:
# Treina modelo final com melhores hiperparâmetros
print("Treinando modelo final com melhores hiperparâmetros...")

# Adiciona parâmetros fixos aos melhores encontrados
best_params = study.best_params.copy()
best_params['scale_pos_weight'] = scale_pos_weight
best_params['random_state'] = 42
best_params['eval_metric'] = 'logloss'

# Treina no conjunto completo de treino
modelo_xgb = xgb.XGBClassifier(**best_params)
modelo_xgb.fit(X_train_scaled, y_train)

print("Modelo final treinado!")

Treinando modelo final com melhores hiperparâmetros...
Modelo final treinado!


In [11]:
# Avalia modelo
y_pred = modelo_xgb.predict(X_test_scaled)
y_proba = modelo_xgb.predict_proba(X_test_scaled)

print("\nResultados:")
print(classification_report(y_test, y_pred, target_names=['Alerta', 'Sonolento']))

print("\nMatriz de Confusão:")
cm = confusion_matrix(y_test, y_pred)
print(f"         Alerta  Sonolento")
print(f"Alerta     {cm[0,0]:4d}      {cm[0,1]:4d}")
print(f"Sonolento  {cm[1,0]:4d}      {cm[1,1]:4d}")

accuracy = (cm[0,0] + cm[1,1]) / cm.sum()
auc_final = roc_auc_score(y_test, y_proba[:, 1])

print(f"\nAcurácia: {accuracy:.3f}")
print(f"AUC: {auc_final:.3f}")


Resultados:
              precision    recall  f1-score   support

      Alerta       0.95      0.96      0.95      4260
   Sonolento       0.92      0.90      0.91      2163

    accuracy                           0.94      6423
   macro avg       0.93      0.93      0.93      6423
weighted avg       0.94      0.94      0.94      6423


Matriz de Confusão:
         Alerta  Sonolento
Alerta     4085       175
Sonolento   216      1947

Acurácia: 0.939
AUC: 0.983


In [12]:
# Salva modelo e componentes
modelo_dir = Path("modelos_xgb_optuna")
modelo_dir.mkdir(exist_ok=True)

# Salva modelo
joblib.dump(modelo_xgb, modelo_dir / "modelo_xgb.joblib")

# Salva scaler
joblib.dump(scaler, modelo_dir / "scaler.joblib")

# Salva estudo Optuna
joblib.dump(study, modelo_dir / "optuna_study.joblib")

# Info das classes e otimização
class_info = {
    "classes": [0, 1],
    "class_names": ["Alerta", "Sonolento"],
    "accuracy": float(accuracy),
    "auc": float(auc_final),
    "best_params": study.best_params,
    "best_auc_validation": float(study.best_value),
    "n_trials": len(study.trials)
}

with open(modelo_dir / "info_classes.json", 'w') as f:
    json.dump(class_info, f, indent=2)

print(f"Modelo salvo em: {modelo_dir}")
print("Arquivos criados:")
print("  - modelo_xgb.joblib")
print("  - scaler.joblib")
print("  - optuna_study.joblib")
print("  - info_classes.json")

Modelo salvo em: modelos_xgb_optuna
Arquivos criados:
  - modelo_xgb.joblib
  - scaler.joblib
  - optuna_study.joblib
  - info_classes.json


In [13]:
# Cria pipeline de predição
pipeline_code = '''import joblib
import numpy as np
import json
from pathlib import Path

class PipelineFadiga:
    def __init__(self, modelo_dir):
        modelo_dir = Path(modelo_dir)
        
        self.extrator = joblib.load(modelo_dir / "extrator.joblib")
        self.scaler = joblib.load(modelo_dir / "scaler.joblib")
        self.modelo = joblib.load(modelo_dir / "modelo_xgb.joblib")
        
        with open(modelo_dir / "info_classes.json", "r") as f:
            self.class_info = json.load(f)
    
    def predict_sequence(self, sequence):
        """Prediz fadiga de uma sequence (90, 4)"""
        sequences = np.expand_dims(sequence, axis=0)
        features = self.extrator.transform(sequences)
        features_scaled = self.scaler.transform(features)
        
        prediction = self.modelo.predict(features_scaled)[0]
        probabilities = self.modelo.predict_proba(features_scaled)[0]
        
        prob_dict = {
            "Alerta": float(probabilities[0]),
            "Sonolento": float(probabilities[1])
        }
        
        class_name = "Alerta" if prediction == 0 else "Sonolento"
        
        return prediction, prob_dict, class_name
'''

with open(modelo_dir / "pipeline.py", 'w') as f:
    f.write(pipeline_code)

print("Pipeline de predição criado: pipeline.py")
print("\nTreino com Optuna concluído! Execute o notebook de teste em tempo real.")

Pipeline de predição criado: pipeline.py

Treino com Optuna concluído! Execute o notebook de teste em tempo real.


In [14]:
# Análise adicional dos resultados do Optuna
print("\n=== ANÁLISE DOS RESULTADOS OPTUNA ===")
print(f"Número total de trials: {len(study.trials)}")
print(f"Melhor trial: {study.best_trial.number}")
print(f"Melhor AUC na validação: {study.best_value:.4f}")
print(f"AUC no conjunto de teste: {auc_final:.4f}")

# Top 5 trials
print("\nTop 5 melhores trials:")
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -1, reverse=True)
for i, trial in enumerate(sorted_trials[:5]):
    if trial.value is not None:
        print(f"  {i+1}. Trial {trial.number}: AUC = {trial.value:.4f}")

print("\nComparação com hiperparâmetros padrão do notebook original:")
print("Original: n_estimators=200, max_depth=6, learning_rate=0.1")
print(f"Optuna: n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}, learning_rate={best_params['learning_rate']:.3f}")


=== ANÁLISE DOS RESULTADOS OPTUNA ===
Número total de trials: 100
Melhor trial: 96
Melhor AUC na validação: 0.9759
AUC no conjunto de teste: 0.9835

Top 5 melhores trials:
  1. Trial 96: AUC = 0.9759
  2. Trial 95: AUC = 0.9757
  3. Trial 93: AUC = 0.9757
  4. Trial 99: AUC = 0.9755
  5. Trial 55: AUC = 0.9753

Comparação com hiperparâmetros padrão do notebook original:
Original: n_estimators=200, max_depth=6, learning_rate=0.1
Optuna: n_estimators=490, max_depth=10, learning_rate=0.055
